<a href="https://colab.research.google.com/github/ZarinRitu32/Myopia_ML/blob/main/SoftCom_02_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SoftCom myopia-risk benchmark - Section 2 - Preprocessing & feature engineering

Replication and extension of Mu et al. 2024 (Front. Med. 11:1482788) on the provided **hybrid cohort (N=1500)**. Pipeline validation only - no clinical claims.

**Pipeline:** SoftCom_01_Data_Validation -> **SoftCom_02_Preprocessing** -> SoftCom_03_Benchmarking -> SoftCom_04_Explainable_AI -> SoftCom_05_Figures -> SoftCom_06_Export

All inputs and outputs live in Google Drive at `MyDrive/SoftCom_Myopia/`. Each notebook checks `artifacts/manifest.json` and stops if an upstream step has not run or its files have changed.

| | |
|---|---|
| **Reads** | outputs of 01 |
| **Writes** | `data/processed/cohort_features.csv`, `train.csv`, `test.csv`, `artifacts/feature_spec.json` |
| **Next** | `SoftCom_03_Benchmarking.ipynb` |

Runtime: Colab CPU is enough. Use *Runtime -> Run all*; accept the Drive prompt in the setup cell. For a quick trial run, set `"FAST_MODE": True` in the CONFIG cell of **every** notebook (fewer CV repeats/bootstraps); set it back to `False` for the reported results.

## 0. Install dependencies and set up Google Drive

In [ ]:
%pip install -q "shap>=0.47" xgboost lightgbm statsmodels imbalanced-learn

In [ ]:
"""Shared setup (identical in every SoftCom notebook).

Mounts Google Drive, defines CONFIG, the Drive folder layout, the run manifest
that links the six notebooks together, seeding, logging and figure helpers.
"""
from __future__ import annotations

import hashlib
import json
import os
import random
import shutil
import sys
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

# ----------------------------------------------------------------------------- CONFIG
CONFIG: Dict[str, Any] = {
    "SEED": 42,
    "TEST_SIZE": 0.3,
    "N_SPLITS": 5,
    "N_REPEATS": 10,
    "N_BOOTSTRAP": 2000,
    "EXPECTED_N_ROWS": 1500,
    "DATA_FILENAME": "full_myopia_hybrid_dataset_N1500.csv",
    "PILOT_FILENAME": "pilot_real.csv",          # optional 73-row pilot; never merged
    "TARGET": "is_myopic",
    "SOURCE_LABEL": "hybrid",
    "COHORT_LABEL": "Hybrid cohort (N=1500) - pipeline validation only, not clinical evidence",
    "DRIVE_PROJECT_DIR": "SoftCom_Myopia",       # folder inside MyDrive holding everything
    "USE_SMOTE": False,                          # SMOTE inside pipeline only (imblearn)
    "USE_LASSO_SELECTION": True,                 # L1-logistic SelectFromModel inside pipeline
    "ABLATION": True,                            # also run FT-Transformer without attention
    "DEEP_MAX_EPOCHS": 200,
    "DEEP_PATIENCE": 20,
    "DEEP_BATCH_SIZE": 64,
    "PERM_N_REPEATS": 20,
    "SHAP_BACKGROUND": 100,
    "SHAP_N_EXPLAIN": 150,
    "SHAP_KERNEL_NSAMPLES": 200,
    "FAST_MODE": False,                          # True = quick trial run (same value in all notebooks)
}
CONFIG["FAST_MODE"] |= os.environ.get("SOFTCOM_FAST", "0") == "1"  # local smoke tests
if CONFIG["FAST_MODE"]:
    CONFIG.update(N_REPEATS=2, N_BOOTSTRAP=200, DEEP_MAX_EPOCHS=25, DEEP_PATIENCE=5,
                  PERM_N_REPEATS=3, SHAP_BACKGROUND=20, SHAP_N_EXPLAIN=20,
                  SHAP_KERNEL_NSAMPLES=60)
SEED: int = CONFIG["SEED"]

NOTEBOOKS: Dict[str, str] = {
    "01_data": "SoftCom_01_Data_Validation.ipynb",
    "02_preprocessing": "SoftCom_02_Preprocessing.ipynb",
    "03_benchmark": "SoftCom_03_Benchmarking.ipynb",
    "04_xai": "SoftCom_04_Explainable_AI.ipynb",
    "05_figures": "SoftCom_05_Figures.ipynb",
    "06_export": "SoftCom_06_Export.ipynb",
}

# ----------------------------------------------------------------------------- Drive
LAUNCH_DIR: Path = Path.cwd()
try:
    from google.colab import drive  # type: ignore

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not Path("/content/drive/MyDrive").is_dir():
        try:
            drive.mount("/content/drive")
        except Exception as exc:  # pragma: no cover - interactive auth
            raise RuntimeError(
                "Google Drive mount failed. Re-run this cell and accept the Drive "
                f"authorisation prompt. Original error: {exc}"
            ) from exc
    PROJECT_ROOT = Path("/content/drive/MyDrive") / CONFIG["DRIVE_PROJECT_DIR"]
else:
    PROJECT_ROOT = Path(os.environ.get("SOFTCOM_ROOT", LAUNCH_DIR / CONFIG["DRIVE_PROJECT_DIR"])).resolve()

DIRS: Dict[str, Path] = {
    name: PROJECT_ROOT / sub
    for name, sub in {
        "raw": "data/raw",
        "processed": "data/processed",
        "models": "models",
        "predictions": "predictions",
        "tables": "tables",
        "figures": "figures",
        "artifacts": "artifacts",
        "logs": "logs",
    }.items()
}
for _d in DIRS.values():
    _d.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_ROOT)  # so ./figures/ and ./tables/ resolve inside Google Drive
MANIFEST_PATH: Path = DIRS["artifacts"] / "manifest.json"


# ----------------------------------------------------------------------------- helpers
def set_global_seed(seed: int = SEED) -> None:
    """Seed python, numpy and (if installed) torch for reproducible runs."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch  # type: ignore

        torch.manual_seed(seed)
        torch.use_deterministic_algorithms(True, warn_only=True)
    except ImportError:
        pass


def log(step: str, message: str) -> None:
    """Print a message and append it to logs/<step>.log on Drive."""
    stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    print(message)
    with open(DIRS["logs"] / f"{step}.log", "a", encoding="utf-8") as fh:
        fh.write(f"[{stamp}] {message}\n")


def sha256_file(path: Path) -> str:
    """Return the SHA-256 hex digest of a file."""
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def rel(path: Path) -> str:
    """Path relative to the project root (as stored in the manifest)."""
    return str(Path(path).resolve().relative_to(PROJECT_ROOT))


def _atomic_write_json(obj: Any, path: Path) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as fh:
        json.dump(obj, fh, indent=2, default=str)
    os.replace(tmp, path)


def save_json(obj: Any, path: Path) -> Path:
    """Write JSON atomically (numpy/pandas values are stringified if needed)."""
    _atomic_write_json(obj, path)
    return path


def load_manifest() -> Dict[str, Any]:
    """Load the cross-notebook run manifest (empty skeleton if absent)."""
    if MANIFEST_PATH.exists():
        with open(MANIFEST_PATH, encoding="utf-8") as fh:
            return json.load(fh)
    return {"project": "SoftCom myopia benchmark", "steps": {}}


def register_outputs(step: str, paths: Iterable[Path], extra: Optional[Dict[str, Any]] = None) -> None:
    """Record a finished notebook step and the SHA-256 of every file it wrote."""
    manifest = load_manifest()
    files = {}
    for p in paths:
        p = Path(p)
        if not p.exists():
            raise FileNotFoundError(f"[{step}] expected output was not written: {p}")
        files[rel(p)] = sha256_file(p)
    manifest["steps"][step] = {
        "notebook": NOTEBOOKS[step],
        "status": "ok",
        "finished_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "fast_mode": CONFIG["FAST_MODE"],
        "outputs": files,
        "extra": extra or {},
    }
    _atomic_write_json(manifest, MANIFEST_PATH)
    print(f"Manifest updated: step '{step}' -> {len(files)} files registered.")


def require_upstream(step: str, needed: Sequence[str]) -> None:
    """Fail loudly unless `step` ran successfully and its files are unchanged.

    `needed` are project-relative paths that this notebook will read.
    """
    manifest = load_manifest()
    info = manifest["steps"].get(step)
    if info is None or info.get("status") != "ok":
        raise RuntimeError(
            f"Upstream step '{step}' has not completed. Open and 'Run all' "
            f"{NOTEBOOKS[step]} first (results folder: {PROJECT_ROOT})."
        )
    for r in needed:
        p = PROJECT_ROOT / r
        if not p.exists():
            raise FileNotFoundError(f"Missing upstream file {r}; re-run {NOTEBOOKS[step]}.")
        expected = info["outputs"].get(r)
        if expected is None:
            raise RuntimeError(f"{r} is not registered by {NOTEBOOKS[step]}; re-run it.")
        if sha256_file(p) != expected:
            raise RuntimeError(
                f"{r} changed after {NOTEBOOKS[step]} finished (hash drift). "
                f"Re-run {NOTEBOOKS[step]} and the notebooks after it."
            )
    if info.get("fast_mode") != CONFIG["FAST_MODE"]:
        print(f"WARNING: {step} ran with FAST_MODE={info.get('fast_mode')}, this notebook uses "
              f"FAST_MODE={CONFIG['FAST_MODE']}.")
    print(f"Upstream OK: {step} ({info['finished_at']}), {len(needed)} files verified.")


sns.set_theme(style="whitegrid", context="notebook", font_scale=1.1)
plt.rcParams.update({"font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11,
                     "legend.fontsize": 10, "savefig.bbox": "tight"})


def save_figure(fig: plt.Figure, name: str, footnote: bool = True) -> List[Path]:
    """Save `fig` as 300-DPI PNG and SVG in ./figures/ (on Drive) and close it."""
    if footnote:  # place below everything already drawn (rotated tick labels included)
        bottom = fig.get_tightbbox(fig.canvas.get_renderer()).y0 / fig.get_figheight()
        fig.text(0.995, min(bottom, 0.0) - 0.01, CONFIG["COHORT_LABEL"], ha="right", va="top", fontsize=8, color="0.45")
    out = []
    try:
        for ext in ("png", "svg"):
            p = DIRS["figures"] / f"{name}.{ext}"
            fig.savefig(p, dpi=300, bbox_inches="tight")
            out.append(p)
    except OSError as exc:
        raise RuntimeError(f"Could not write figure {name} to {DIRS['figures']}: {exc}") from exc
    finally:
        plt.close(fig)
    print(f"Saved figure: figures/{name}.png + .svg")
    return out


def show_saved(name: str) -> None:
    """Display a saved PNG inline (Colab/Jupyter only)."""
    try:
        from IPython.display import Image, display  # type: ignore

        if "ipykernel" in sys.modules:
            display(Image(filename=str(DIRS["figures"] / f"{name}.png"), width=720))
    except ImportError:
        pass


set_global_seed(SEED)
print(f"Colab: {IN_COLAB} | results folder: {PROJECT_ROOT} | FAST_MODE={CONFIG['FAST_MODE']}")

Mounted at /content/drive
Colab: True | results folder: /content/drive/MyDrive/SoftCom_Myopia | FAST_MODE=False


## 2.1 Load the validated cohort from notebook 01

In [ ]:
STEP = "02_preprocessing"
require_upstream("01_data", ["data/processed/cohort_validated.csv"])
df = pd.read_csv(DIRS["processed"] / "cohort_validated.csv")
TARGET = CONFIG["TARGET"]
log(STEP, f"Main cohort: {len(df)} rows (source={df['source'].unique().tolist()})")

PILOT_PATH = DIRS["raw"] / CONFIG["PILOT_FILENAME"]
pilot: Optional[pd.DataFrame] = None
if PILOT_PATH.exists():
    pilot = pd.read_csv(PILOT_PATH).assign(source="pilot")
    log(STEP, f"Pilot cohort: {len(pilot)} rows - kept separate, never concatenated with the main cohort")
else:
    log(STEP, "Pilot cohort: 0 rows (no pilot file) - main cohort only")

Upstream OK: 01_data (2026-09-25T16:12:44+00:00), 1 files verified.
Main cohort: 1500 rows (source=['hybrid'])
Pilot cohort: 0 rows (no pilot file) - main cohort only


## 2.2 Feature engineering
* `corneal_radius_mm = 337.5 / corneal_power_d` (keratometric index 1.3375)
* `al_cr_ratio = axial_length_mm / corneal_radius_mm`
* `near_work_hours = screen_hours_daily + continuous_reading_min / 60`
* `near_outdoor_ratio = near_work_hours / (outdoor_hours_daily + 0.25)` (0.25 h offset avoids division by zero)
* `myopia_severity` (0 none, 1 mild -3<=SE<-0.5, 2 moderate -6<=SE<-3, 3 high SE<-6) is descriptive only.

In [ ]:
ORDINAL_MAPS: Dict[str, Dict[int, int]] = {
    "gender": {0: 0, 1: 1},
    "parental_myopia_count": {0: 0, 1: 1, 2: 2},
    "diet_sugary_freq": {0: 0, 1: 1},
}
NOMINAL_FEATURES: List[str] = []  # none in this dataset; one-hot(drop_first) is applied if added


def engineer_features(data: pd.DataFrame) -> pd.DataFrame:
    """Add derived biometric / lifestyle features and apply explicit ordinal maps."""
    out = data.copy()
    out["corneal_radius_mm"] = 337.5 / out["corneal_power_d"]
    out["al_cr_ratio"] = out["axial_length_mm"] / out["corneal_radius_mm"]
    out["near_work_hours"] = out["screen_hours_daily"] + out["continuous_reading_min"] / 60.0
    out["near_outdoor_ratio"] = out["near_work_hours"] / (out["outdoor_hours_daily"] + 0.25)
    se = out["spherical_equivalent_d"]
    out["myopia_severity"] = np.select([se < -6, se < -3, se <= -0.5], [3, 2, 1], default=0)
    for col, mapping in ORDINAL_MAPS.items():
        unknown = sorted(set(out[col].dropna().unique()) - set(mapping))
        if unknown:
            raise ValueError(f"{col}: values {unknown} are not in ORDINAL_MAPS - update the map explicitly.")
        out[col] = out[col].map(mapping).astype(int)
    if NOMINAL_FEATURES:
        out = pd.get_dummies(out, columns=NOMINAL_FEATURES, drop_first=True, dtype=int)
    return out


dfe = engineer_features(df)
print(dfe[["corneal_radius_mm", "al_cr_ratio", "near_work_hours", "near_outdoor_ratio"]].describe().round(3).T)
print("Severity counts:", dfe["myopia_severity"].value_counts().sort_index().to_dict())

                     count   mean    std    min    25%    50%    75%     max
corneal_radius_mm   1500.0  7.818  0.213  7.181  7.668  7.808  7.958   8.616
al_cr_ratio         1500.0  3.087  0.124  2.642  3.006  3.089  3.167   3.522
near_work_hours     1500.0  4.715  2.628  0.767  2.762  4.033  5.917  14.000
near_outdoor_ratio  1500.0  5.555  5.246  0.189  2.152  3.929  7.440  40.267
Severity counts: {0: 1032, 1: 375, 2: 88, 3: 5}


## 2.3 Feature set and leakage exclusions

In [ ]:
NUMERIC_FEATURES = ["age", "screen_hours_daily", "continuous_reading_min", "outdoor_hours_daily",
                    "axial_length_mm", "corneal_power_d", "anterior_chamber_depth", "al_cr_ratio",
                    "near_outdoor_ratio"]
CATEGORICAL_FEATURES = list(ORDINAL_MAPS)  # integer codes; embedded by the deep models
EXCLUDED = {
    "spherical_equivalent_d": "defines the label (is_myopic == SE <= -0.5 D) - target leakage",
    "myopia_severity": "derived from SE - target leakage",
    "corneal_radius_mm": "deterministic transform of corneal_power_d (collinear)",
    "near_work_hours": "numerator of near_outdoor_ratio; raw components already included",
    "row_id": "identifier", "source": "provenance tag",
}
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
cards = [int(max(ORDINAL_MAPS[c].values()) + 1) for c in CATEGORICAL_FEATURES]
assert set(FEATURES).isdisjoint(EXCLUDED), "An excluded column leaked into FEATURES"
assert dfe[FEATURES].shape[1] == len(FEATURES)
print(f"{len(FEATURES)} model features: {FEATURES}")
for k, v in EXCLUDED.items():
    print(f"  excluded {k:<24} {v}")

12 model features: ['age', 'screen_hours_daily', 'continuous_reading_min', 'outdoor_hours_daily', 'axial_length_mm', 'corneal_power_d', 'anterior_chamber_depth', 'al_cr_ratio', 'near_outdoor_ratio', 'gender', 'parental_myopia_count', 'diet_sugary_freq']
  excluded spherical_equivalent_d   defines the label (is_myopic == SE <= -0.5 D) - target leakage
  excluded myopia_severity          derived from SE - target leakage
  excluded corneal_radius_mm        deterministic transform of corneal_power_d (collinear)
  excluded near_work_hours          numerator of near_outdoor_ratio; raw components already included
  excluded row_id                   identifier
  excluded source                   provenance tag


## 2.4 Stratified 70/30 holdout split (paper replica)
Imputation, scaling, LASSO selection and optional SMOTE are **not** fitted here; they live
inside each model pipeline in notebook 03 so they are refitted within every CV fold.

In [ ]:
from sklearn.model_selection import train_test_split

train_idx, test_idx = train_test_split(dfe.index.values, test_size=CONFIG["TEST_SIZE"], stratify=dfe[TARGET],
                                       random_state=SEED)
dfe["split"] = "train"
dfe.loc[test_idx, "split"] = "test"
train_df, test_df = dfe.loc[train_idx].sort_values("row_id"), dfe.loc[test_idx].sort_values("row_id")
split_hash = hashlib.sha256(",".join(map(str, sorted(test_df.row_id))).encode()).hexdigest()
print(f"train={len(train_df)} (prev {train_df[TARGET].mean():.3f}) | test={len(test_df)} "
      f"(prev {test_df[TARGET].mean():.3f}) | split hash {split_hash[:12]}")

train=1050 (prev 0.312) | test=450 (prev 0.311) | split hash d9a3850aef85


In [ ]:
paths = {
    "features_all": DIRS["processed"] / "cohort_features.csv",
    "train": DIRS["processed"] / "train.csv",
    "test": DIRS["processed"] / "test.csv",
    "spec": DIRS["artifacts"] / "feature_spec.json",
}
spec = {
    "target": TARGET, "numeric": NUMERIC_FEATURES, "categorical": CATEGORICAL_FEATURES,
    "cat_cardinalities": cards, "ordinal_maps": {k: {str(a): b for a, b in v.items()} for k, v in ORDINAL_MAPS.items()},
    "nominal_one_hot": NOMINAL_FEATURES, "excluded": EXCLUDED, "split_hash": split_hash,
    "n_train": len(train_df), "n_test": len(test_df), "n_pilot": 0 if pilot is None else len(pilot),
}
try:
    dfe.to_csv(paths["features_all"], index=False)
    train_df.to_csv(paths["train"], index=False)
    test_df.to_csv(paths["test"], index=False)
    save_json(spec, paths["spec"])
    if pilot is not None:
        pp = DIRS["processed"] / "pilot_features.csv"
        engineer_features(pilot).to_csv(pp, index=False)
        paths["pilot"] = pp
except OSError as exc:
    raise RuntimeError(f"Writing to Google Drive failed: {exc}") from exc
register_outputs(STEP, paths.values(), {"n_features": len(FEATURES), "split_hash": split_hash})

Manifest updated: step '02_preprocessing' -> 4 files registered.


In [ ]:
print("=" * 70)
print(f"02 DONE | features={len(FEATURES)} | train={len(train_df)} | test={len(test_df)} | "
      f"pilot={spec['n_pilot']}")
print(f"NEXT -> open {NOTEBOOKS['03_benchmark']} and Run all")

02 DONE | features=12 | train=1050 | test=450 | pilot=0
NEXT -> open SoftCom_03_Benchmarking.ipynb and Run all
